Загружаем готовый датасет с отзывами на Яндекс Картах за 2023 год

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df = pd.read_csv('geo-reviews-dataset-2023.csv')


Saving geo-reviews-dataset-2023.csv to geo-reviews-dataset-2023.csv


In [ ]:
df.head()
df.info()
df.describe(include='all')
print(f"Строк: {df.shape[0]}, Колонок: {df.shape[1]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   address  500000 non-null  object 
 1   name_ru  499030 non-null  object 
 2   rating   500000 non-null  float64
 3   rubrics  500000 non-null  object 
 4   text     500000 non-null  object 
dtypes: float64(1), object(4)
memory usage: 19.1+ MB
Строк: 500000, Колонок: 5


Смотрим какие рубрики встречаются чаще всего и выбираем топ три для дальнейшей работы

In [ ]:
print(df['rubrics'].head(10))
import ast
from collections import Counter

def extract_rubrics(rubric_str):
    try:

        return ast.literal_eval(rubric_str)
    except:

        return [x.strip() for x in str(rubric_str).split(',')]

all_rubrics = []
for rub in df['rubrics'].dropna():
    all_rubrics.extend(extract_rubrics(rub))

rubric_counts = Counter(all_rubrics)

print("\nТоп-30 рубрик:")
for rubric, count in rubric_counts.most_common(30):
    print(f"  {rubric}: {count}")
print(f"\nВсего уникальных рубрик: {len(rubric_counts)}")


0                                       Жилой комплекс
1    Магазин продуктов;Продукты глубокой заморозки;...
2                                          Фитнес-клуб
3          Пункт проката;Прокат велосипедов;Сапсёрфинг
4    Салон красоты;Визажисты, стилисты;Салон бровей...
5            Оператор сотовой связи;Интернет-провайдер
6                                                 Кафе
7    Вейп-шоп;Магазин табака и курительных принадле...
8                                         Кафе;Кофейня
9                                         Кафе;Кофейня
Name: rubrics, dtype: object

Топ-30 рубрик:
  Гостиница: 42242
  Ресторан: 14615
  Кафе: 12366
  Супермаркет: 8899
  Бар: 7528
  Ресторан;Бар: 6280
  Автосервис: 6206
  Медцентр: 5904
  паб: 5572
  Магазин продуктов: 5289
  Музей: 5005
  Быстрое питание: 4902
  Ресторан;Кафе: 4554
  Супермаркет;Магазин продуктов: 4541
  Пункт выдачи: 4316
  Аптека: 4114
  автотехцентр: 3805
  Парк культуры и отдыха: 3777
  Достопримечательность: 3760
  База: 375

Удаляем адрес, чтобы облегчить датасет

In [ ]:
print(df.columns.tolist())

['address', 'name_ru', 'rating', 'rubrics', 'text']


In [ ]:
df = df.drop(columns=['address'])
print(df.columns.tolist())

['name_ru', 'rating', 'rubrics', 'text']


In [ ]:
import pandas as pd

selected_categories = ['Гостиница', 'Ресторан', 'Кафе']

# Фильтруем
df_filtered = df[df['rubrics'].fillna('').str.lower().str.contains('|'.join(selected_categories).lower())].copy()

print(f"Исходно: {len(df)} строк")
print(f"Отфильтровано: {len(df_filtered)} строк")
print(f"Колонки: {df_filtered.columns.tolist()}")

# Сохраняем
df_filtered.to_csv('filtered_hotels_restaurants_cafes.csv', index=False)
print("✅ Сохранено как 'filtered_hotels_restaurants_cafes.csv'")

# Проверим, что сохранилось
import os
print(f"Размер файла: {os.path.getsize('filtered_hotels_restaurants_cafes.csv') / 1024 / 1024:.2f} MB")

Исходно: 500000 строк
Отфильтровано: 136421 строк
Колонки: ['name_ru', 'rating', 'rubrics', 'text']
✅ Сохранено как 'filtered_hotels_restaurants_cafes.csv'
Размер файла: 84.41 MB


In [ ]:
from google.colab import files
files.download('filtered_hotels_restaurants_cafes.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Создаём золотой стандарт из 3500 сгенерированных рекламных  отзывов и 5000 реальных отзывов с датасета и загружаем его

In [ ]:
import pandas as pd
import random
from tqdm import tqdm

# Загружаем ваш файл с 136 000 строк
df_filtered = pd.read_csv('filtered_hotels_restaurants_cafes.csv')
print(f"✅ Загружено {len(df_filtered)} строк")
print(f"Колонки: {df_filtered.columns.tolist()}")

fake_templates = []

# Категории рекламных текстов
templates_by_type = {
    'восторженные': [
        "Очень крутое место! Всем рекомендую! {name} просто топ!",
        "{name} — это любовь с первого визита! Обалденно!",
        "Лучшее открытие этого года! {name} просто бомба!",
        "Я в восторге! {name} задаёт тренды в {city}!",
        "Шикарно! {name} превзошёл все ожидания!"
    ],
    'призывные': [
        "Срочно бегите в {name}! Лучший {category} в городе!",
        "Обязательно к посещению! {name} ждёт вас!",
        "Не проходите мимо {name}! Это что-то невероятное!",
        "Запишитесь прямо сейчас! {name} — топ заведение!",
        "Всем друзьям советую {name}! Не пожалеете!"
    ],
    'скидочные': [
        "Только сегодня скидки в {name}! Успейте!",
        "Акция! В {name} скидка 20% на всё меню!",
        "Спецпредложение в {name}: второй заказ в подарок!",
        "Купоны на скидку в {name}! Забирайте!",
        "Приведи друга в {name} и получи скидку 50%!"
    ],
    'эмоциональные': [
        "Вау! {name} — это невероятно круто! Топчик!",
        "Огонь! {name} просто зажигает! Всем советую!",
        "Класс! {name} суперское место! Оценка 5+!",
        "{name} — отпад! Лучшее вложение денег!",
        "{name} рулит! Обязательно вернусь!"
    ],
    'сравнительные': [
        "Лучше {name} нет ничего в {city}!",
        "{name} — топ среди всех {category}!",
        "По сравнению с другими, {name} просто небо и земля!",
        "Самый лучший {category} — это {name}!",
        "{name} вне конкуренции! Всем рекомендую!"
    ],
    'короткие': [
        "{name} — супер! 10/10!",
        "{name} огонь! 🔥",
        "{name} — мастхев!",
        "{name} лучший!",
        "{name} просто сказка!"
    ]
}

# Собираем все шаблоны
for category, templates in templates_by_type.items():
    fake_templates.extend(templates)

expanded_templates = []
for template in fake_templates:
    expanded_templates.append(template)
    # Вариации с разными знаками препинания
    expanded_templates.append(template.replace("!", "!!!"))
    expanded_templates.append(template.replace("!", "!!!!"))
    expanded_templates.append(template + " 🔥🔥🔥")

fake_templates = expanded_templates * 5

print(f"Сгенерировано {len(fake_templates)} шаблонов рекламы")

real_names = df_filtered['name_ru'].dropna().unique().tolist()
real_categories = df_filtered['rubrics'].dropna().unique().tolist()
real_cities = df_filtered['city'].dropna().unique().tolist() if 'city' in df_filtered.columns else ['Москва', 'СПБ', 'Казань']

print(f"Уникальных заведений: {len(real_names)}")
print(f"Уникальных рубрик: {len(real_categories)}")

num_fake = 3500
fake_reviews = []

for i in tqdm(range(num_fake), desc="Генерация рекламных отзывов"):
    template = random.choice(fake_templates)
    name = random.choice(real_names)
    category = random.choice(real_categories)
    city = random.choice(real_cities) if real_cities else "Москва"

    fake_text = template.format(name=name, category=category, city=city)

    # Добавляем случайные хэштеги
    if random.random() > 0.7:
        fake_text += f" #{random.choice(['рекомендую', 'скидки', 'акции', 'топ', 'лучшее'])}"

    fake_reviews.append({
        'text': fake_text,
        'rating': random.choice([4, 4.5, 5]),  # высокие оценки
        'name_ru': name,
        'rubrics': category,
        'city': city,
        'is_fake': 1
    })

df_fakes = pd.DataFrame(fake_reviews)
print(f"✅ Создано {len(df_fakes)} рекламных отзывов")

num_real = 5000
df_real = df_filtered.sample(n=min(num_real, len(df_filtered)), random_state=42).copy()
df_real['is_fake'] = 0

if 'review' in df_real.columns and 'text' not in df_real.columns:
    df_real = df_real.rename(columns={'review': 'text'})

print(f"✅ Взято {len(df_real)} реальных отзывов")

df_mixed = pd.concat([df_real, df_fakes], ignore_index=True)
df_mixed = df_mixed.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n" + "="*50)
print("ИТОГИ СОЗДАНИЯ ЗОЛОТОГО СТАНДАРТА:")
print("="*50)
print(f"Реальных отзывов (is_fake=0): {len(df_mixed[df_mixed['is_fake']==0])}")
print(f"Рекламных фейков (is_fake=1): {len(df_mixed[df_mixed['is_fake']==1])}")
print(f"Всего размеченных отзывов: {len(df_mixed)}")
print(f"Доля рекламы: {df_mixed['is_fake'].mean()*100:.1f}%")

output_file = 'золотой_стандарт_8500.xlsx'
df_mixed.to_excel(output_file, index=False)
print(f"\n✅ Золотой стандарт сохранён как '{output_file}'")

print("\n" + "="*50)
print("ПРИМЕРЫ СОЗДАННЫХ ОТЗЫВОВ:")
print("="*50)

print("\n🔴 ПРИМЕРЫ РЕКЛАМЫ (is_fake=1):")
for i, row in df_fakes.head(5).iterrows():
    print(f"  - {row['text'][:100]}...")

print("\n🟢 ПРИМЕРЫ РЕАЛЬНЫХ ОТЗЫВОВ (is_fake=0):")
for i, row in df_real.head(5).iterrows():
    text = row['text'][:100] if len(str(row['text'])) > 100 else row['text']
    print(f"  - {text}...")

✅ Загружено 136421 строк
Колонки: ['name_ru', 'rating', 'rubrics', 'text']
Сгенерировано 600 шаблонов рекламы
Уникальных заведений: 30312
Уникальных рубрик: 2481


Генерация рекламных отзывов: 100%|██████████| 3500/3500 [00:00<00:00, 156097.83it/s]

✅ Создано 3500 рекламных отзывов
✅ Взято 5000 реальных отзывов

ИТОГИ СОЗДАНИЯ ЗОЛОТОГО СТАНДАРТА:
Реальных отзывов (is_fake=0): 5000
Рекламных фейков (is_fake=1): 3500
Всего размеченных отзывов: 8500
Доля рекламы: 41.2%



✅ Золотой стандарт сохранён как 'золотой_стандарт_8500.xlsx'

ПРИМЕРЫ СОЗДАННЫХ ОТЗЫВОВ:

🔴 ПРИМЕРЫ РЕКЛАМЫ (is_fake=1):
  - Очень крутое место!!!! Всем рекомендую!!!! Kitchen&food просто топ!!!!...
  - По сравнению с другими, АнапаГрад просто небо и земля!!!!...
  - Самый лучший Суши-бар;Бар, паб;Кафе;Ресторан — это РОД_Кошелев!!!...
  - По сравнению с другими, Голубино просто небо и земля!!! #акции...
  - Лучшее открытие этого года! Грейс Круиз просто бомба! 🔥🔥🔥...

🟢 ПРИМЕРЫ РЕАЛЬНЫХ ОТЗЫВОВ (is_fake=0):
  - Отличное завершение\nПрекрасная кухня\nСвежее пиво \nЗамечательное обслуживание\nЖивую музыку мы не ...
  - Посетили ресторан  2 раза. Очень понравилось. Все очень вкусно. Роллы потрясающие.  Мидии в сливочно...
  - Очень хорошее заведение с отличной кухней и обслуживанием. Шашлык никаких нареканий не вызывает. Кро...
  - Очень вкусная еда, приятная хозяйка, пошла на встречу при полной посадке нашла для нас местечко 😊 ос...
  - Отдыхали у Андрея иТатьяны много раз. Есть всё для

In [ ]:
from google.colab import files
files.download('золотой_стандарт_8500.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Переходим к обучению модели, создаём признаки

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import re
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

print("="*60)
print("ШАГ 1: Загрузка золотого стандарта")
print("="*60)

df = pd.read_excel('золотой_стандарт_8500.xlsx')
print(f"✅ Загружено размеченных отзывов: {len(df)}")
print(f"   - Реальных (0): {len(df[df['is_fake']==0])}")
print(f"   - Реклама (1): {len(df[df['is_fake']==1])}")
print(f"   - Доля рекламы: {df['is_fake'].mean()*100:.1f}%")

print("\n" + "="*60)
print("ШАГ 2: Предобработка текста")
print("="*60)

def clean_text(text):
    """Очистка текста"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^а-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].astype(str).apply(clean_text)
print("✅ Тексты очищены")

print("\n" + "="*60)
print("ШАГ 3: Векторизация (TF-IDF)")
print("="*60)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(df['clean_text'])
y = df['is_fake'].values

print(f"Размер матрицы признаков: {X.shape}")
print(f"Количество признаков: {X.shape[1]}")

# Разделение на train/test
print("\n" + "="*60)
print("ШАГ 4: Разделение на train (80%) и test (20%)")
print("="*60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train размер: {X_train.shape[0]} строк")
print(f"Test размер: {X_test.shape[0]} строк")
print(f"\nРаспределение в train:")
print(f"  - Реальных: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"  - Реклама: {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")

#Обучение модели
print("\n" + "="*60)
print("ШАГ 5: Обучение модели")
print("="*60)

# Попробуем разные модели
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'Linear SVM': LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
}

best_model = None
best_accuracy = 0

for name, model in models.items():
    print(f"\nОбучаем {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"  Точность: {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        print(f"  ✅ Новая лучшая модель!")

print(f"\n🏆 Лучшая модель: {type(best_model).__name__} с точностью {best_accuracy:.4f}")

# Оценка качества
print("\n" + "="*60)
print("ШАГ 6: Детальная оценка лучшей модели")
print("="*60)

y_pred = best_model.predict(X_test)

print(f"\nТочность (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print(f"\nДетальный отчёт:")
print(classification_report(y_test, y_pred, target_names=['Реальный (0)', 'Реклама (1)']))

print("\nМатрица ошибок:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=['Реальный', 'Реклама'], columns=['Предсказано 0', 'Предсказано 1']))

print("\n" + "="*60)
print("ШАГ 7: Применяем модель к полному датасету (136 000 отзывов)")
print("="*60)

# Загружаем полный датасет
df_full = pd.read_csv('filtered_hotels_restaurants_cafes.csv')
print(f"Загружено полных отзывов: {len(df_full)}")

if 'review' in df_full.columns and 'text' not in df_full.columns:
    df_full = df_full.rename(columns={'review': 'text'})

# Очищаем текст
print("Очищаем тексты...")
df_full['clean_text'] = df_full['text'].astype(str).apply(clean_text)

# Векторизуем
print("Векторизуем...")
X_full = vectorizer.transform(df_full['clean_text'])

# Предсказываем
print("Предсказываем метки...")
df_full['is_fake_predicted'] = best_model.predict(X_full)
df_full['fake_probability'] = best_model.predict_proba(X_full)[:, 1] if hasattr(best_model, 'predict_proba') else 0

# Статистика предсказаний
print(f"\nРезультаты предсказаний для 136 000 отзывов:")
print(f"  - Предсказано реальных (0): {(df_full['is_fake_predicted'] == 0).sum()} ({(df_full['is_fake_predicted'] == 0).mean()*100:.1f}%)")
print(f"  - Предсказано реклама (1): {(df_full['is_fake_predicted'] == 1).sum()} ({(df_full['is_fake_predicted'] == 1).mean()*100:.1f}%)")


print("\n" + "="*60)
print("ШАГ 8: Сохранение результатов")
print("="*60)

joblib.dump(best_model, 'fake_detector_final.pkl')
joblib.dump(vectorizer, 'vectorizer_final.pkl')
print("✅ Модель сохранена")

# Сохраняем полный датасет с предсказаниями
df_full.to_csv('full_dataset_with_predictions.csv', index=False)
print("✅ Полный датасет с предсказаниями сохранён")

# Сохраняем только рекламные отзывы
df_ads = df_full[df_full['is_fake_predicted'] == 1]
df_ads.to_csv('advertisements_detected.csv', index=False)
print(f"✅ Сохранено {len(df_ads)} отзывов, предсказанных как реклама")

print("\n" + "="*60)
print("ПРИМЕРЫ ПРЕДСКАЗАНИЙ МОДЕЛИ:")
print("="*60)

# Отзывы, предсказанные как реклама
print("\n🔴 ОТЗЫВЫ, ПРЕДСКАЗАННЫЕ КАК РЕКЛАМА (вероятность > 70%):")
high_prob = df_full[df_full['fake_probability'] > 0.7].head(5)
for i, row in high_prob.iterrows():
    text = row['text'][:100] if len(str(row['text'])) > 100 else row['text']
    print(f"\n  Текст: {text}...")
    print(f"  Вероятность рекламы: {row['fake_probability']:.2%}")

# Отзывы, предсказанные как реальные
print("\n🟢 ОТЗЫВЫ, ПРЕДСКАЗАННЫЕ КАК РЕАЛЬНЫЕ (вероятность < 30%):")
low_prob = df_full[df_full['fake_probability'] < 0.3].head(5)
for i, row in low_prob.iterrows():
    text = row['text'][:100] if len(str(row['text'])) > 100 else row['text']
    print(f"\n  Текст: {text}...")
    print(f"  Вероятность рекламы: {row['fake_probability']:.2%}")

ШАГ 1: Загрузка золотого стандарта
✅ Загружено размеченных отзывов: 8500
   - Реальных (0): 5000
   - Реклама (1): 3500
   - Доля рекламы: 41.2%

ШАГ 2: Предобработка текста
✅ Тексты очищены

ШАГ 3: Векторизация (TF-IDF)
Размер матрицы признаков: (8500, 10000)
Количество признаков: 10000

ШАГ 4: Разделение на train (80%) и test (20%)
Train размер: 6800 строк
Test размер: 1700 строк

Распределение в train:
  - Реальных: 4000 (58.8%)
  - Реклама: 2800 (41.2%)

ШАГ 5: Обучение модели

Обучаем Logistic Regression...
  Точность: 0.9994
  ✅ Новая лучшая модель!

Обучаем Random Forest...
  Точность: 0.9994

Обучаем Linear SVM...
  Точность: 1.0000
  ✅ Новая лучшая модель!

🏆 Лучшая модель: LinearSVC с точностью 1.0000

ШАГ 6: Детальная оценка лучшей модели

Точность (Accuracy): 1.0000

Детальный отчёт:
              precision    recall  f1-score   support

Реальный (0)       1.00      1.00      1.00      1000
 Реклама (1)       1.00      1.00      1.00       700

    accuracy                 

In [ ]:
print("="*60)
print("🔴 ВСЕ ФЕЙКОВЫЕ/РЕКЛАМНЫЕ ОТЗЫВЫ")
print("="*60)

# Получаем все отзывы, предсказанные как реклама
fake_reviews = df_full[df_full['is_fake_predicted'] == 1].copy()

# Сортируем по вероятности рекламы (от самых явных)
fake_reviews = fake_reviews.sort_values('fake_probability', ascending=False)

print(f"\nВсего обнаружено рекламных отзывов: {len(fake_reviews)}")
print(f"Это {len(fake_reviews)/len(df_full)*100:.2f}% от всех отзывов\n")

# Выводим все фейковые отзывы
for i, (idx, row) in enumerate(fake_reviews.iterrows(), 1):
    print(f"{'='*60}")
    print(f"📢 РЕКЛАМНЫЙ ОТЗЫВ #{i}")
    print(f"{'='*60}")
    print(f"Заведение: {row.get('name_ru', 'Не указано')}")
    print(f"Рубрика: {row.get('rubrics', 'Не указано')}")
    if 'rating' in row:
        print(f"Оценка: {row['rating']}")
    print(f"Вероятность рекламы: {row['fake_probability']:.2%}")
    print(f"\nТекст отзыва:")
    print(f"{row['text']}")
    print()

# Сохраняем все фейки в отдельный файл
fake_reviews.to_excel('all_fake_reviews.xlsx', index=False)
print(f"\n✅ Все {len(fake_reviews)} рекламных отзывов сохранены в 'all_fake_reviews.xlsx'")

Мы видем, что модель работает неправильно. Надо улучшать золотой стандарт

In [ ]:
# Улучшение золотого стандарта
import pandas as pd
import random

# Загружаем существующий золотой стандарт
df = pd.read_excel('Датасет_с_рекламой.xlsx')
print(f"Текущий размер золотого стандарта: {len(df)}")
print(f"Реклама: {df['is_fake'].sum()}, Реальных: {len(df)-df['is_fake'].sum()}")

real_positive_reviews = [
    "Супер заведение, супер вкусно и супер вкусные напитки. Подача тоже огонь. Однозначно рекомендую сюда попасть. Были в пятницу вечером - официант Андрей Андреевич - просто бомба.",
    "Вау! Это лучшее место, где я был! Очень вкусно, атмосфера шикарная, персонал вежливый. Всем рекомендую! Приду ещё не раз!",
    "Потрясающе! Блюда просто шедевральные, особенно фирменный салат. Обслуживание на высоте. Советую всем!",
    "Классное место! Очень вкусная кухня, приятные цены. Официанты доброжелательные. Обязательно вернусь!"
]

# Отзывы, которые СКРЫТАЯ РЕКЛАМА (должны быть 1)
subtle_fake_reviews = [
    "Были в пятницу вечером. Скидка 20% по промокоду. Вкусно, но цены высоковаты.",
    "Хорошее место. При заказе от 3000 руб - десерт в подарок. Акция действует до конца месяца.",
    "Очень вкусно! Особенно по понедельникам скидка 50% на пиццу. Успейте!",
    "Неплохо. Есть бизнес-ланч за 350 руб. Приходите в будни."
]

# Добавляем их в датасет с правильными метками
new_rows = []
for text in real_positive_reviews:
    new_rows.append({
        'text': text,
        'is_fake': 0,
        'rating': 5,
        'name_ru': 'Тестовое заведение',
        'rubrics': 'Ресторан'
    })

for text in subtle_fake_reviews:
    new_rows.append({
        'text': text,
        'is_fake': 1,
        'rating': 4,
        'name_ru': 'Тестовое заведение',
        'rubrics': 'Кафе'
    })

df_improved = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

print(f"\n✅ Добавлено {len(new_rows)} сложных примеров")
print(f"Новый размер золотого стандарта: {len(df_improved)}")
print(f"Реклама: {df_improved['is_fake'].sum()}, Реальных: {len(df_improved)-df_improved['is_fake'].sum()}")

# Сохраняем в тот же файл (перезаписываем)
df_improved.to_excel('Датасет_с_рекламой.xlsx', index=False)
print("\n✅ Золотой стандарт обновлён и сохранён в 'Датасет_с_рекламой.xlsx'")

Текущий размер золотого стандарта: 600
Реклама: 100, Реальных: 500

✅ Добавлено 8 сложных примеров
Новый размер золотого стандарта: 608
Реклама: 104, Реальных: 504

✅ Золотой стандарт обновлён и сохранён в 'Датасет_с_рекламой.xlsx'


In [ ]:
from google.colab import files
files.download('Датасет_с_рекламой.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Мы выбираем более сложную модель RuBERT, языковую модель для обработки текстов на русском языке

In [ ]:
!pip install transformers datasets torch accelerate scikit-learn pandas openpyxl python-telegram-bot --quiet

print("✅ Библиотеки установлены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 10.1 MB/s eta 0:00:00
✅ Библиотеки установлены


In [ ]:
import pandas as pd
import random

# Загружаем существующий золотой стандарт (если есть)
try:
    df = pd.read_excel('Датасет_с_рекламой.xlsx')
    print(f"Загружен существующий датасет: {len(df)} строк")
except:
    df_filtered = pd.read_csv('filtered_hotels_restaurants_cafes.csv')
    if 'review' in df_filtered.columns:
        df_filtered = df_filtered.rename(columns={'review': 'text'})
    df = df_filtered.sample(n=500, random_state=42).copy()
    df['is_fake'] = 0
    print(f"Создан базовый датасет из реальных отзывов: {len(df)} строк")

print("\nГенерируем рекламные отзывы...")

# Расширенные шаблоны рекламы
fake_templates = [
    "Очень крутое место! Всем рекомендую! {name} просто топ!",
    "Лучший {category} в городе! Обязательно посетите! Акции каждый день!",
    "Отличный сервис! Приходите в {name} — не пожалеете! Скидки постоянным клиентам!",
    "Восторг! {name} — это любовь с первого визита. Самое лучшее место!",
    "Потрясающе! Обязательно к посещению! {name} задаёт тренды!",
    "Скидки до 50% в {name}! Только до конца месяца! Успейте!",
    "Бесплатная доставка при заказе от 1000₽ в {name}! Акция!",
    "{name} — твоя love story с первого кусочка пиццы! ❤️",
    "Срочно! {name} дарит подарки всем гостям на этой неделе!",
    "Только сегодня: скидка 30% на всё меню в {name}!",
]

if 'name_ru' in df.columns:
    real_names = df[df['is_fake']==0]['name_ru'].dropna().unique().tolist()
    real_categories = df['rubrics'].dropna().unique().tolist() if 'rubrics' in df.columns else ['Ресторан', 'Кафе', 'Гостиница']
else:
    real_names = ['Заведение', 'Ресторан', 'Кафе']
    real_categories = ['Ресторан', 'Кафе']

# Генерируем 3500 фейков
fake_reviews = []
for i in range(3500):
    template = random.choice(fake_templates)
    name = random.choice(real_names) if real_names else "наше заведение"
    category = random.choice(real_categories) if real_categories else "ресторан"

    fake_text = template.format(name=name, category=category)

    fake_reviews.append({
        'text': fake_text,
        'rating': random.choice([4, 4.5, 5]),
        'name_ru': name if real_names else None,
        'rubrics': category if real_categories else None,
        'is_fake': 1
    })

df_fakes = pd.DataFrame(fake_reviews)

# Берём 5000 реальных отзывов
df_real = df[df['is_fake'] == 0].sample(n=min(5000, len(df[df['is_fake']==0])), random_state=42).copy()

# Объединяем
df_combined = pd.concat([df_real, df_fakes], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Сохраняем
df_combined.to_excel('golden_standard.xlsx', index=False)

print(f"\n{'='*50}")
print(f"ЗОЛОТОЙ СТАНДАРТ СОЗДАН:")
print(f"{'='*50}")
print(f"Реальных отзывов (is_fake=0): {len(df_combined[df_combined['is_fake']==0])}")
print(f"Рекламных отзывов (is_fake=1): {len(df_combined[df_combined['is_fake']==1])}")
print(f"Всего: {len(df_combined)}")
print(f"✅ Сохранён как 'golden_standard.xlsx'")

Загружен существующий датасет: 608 строк

Генерируем рекламные отзывы...

ЗОЛОТОЙ СТАНДАРТ СОЗДАН:
Реальных отзывов (is_fake=0): 504
Рекламных отзывов (is_fake=1): 3500
Всего: 4004
✅ Сохранён как 'golden_standard.xlsx'


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Загружаем золотой стандарт
df = pd.read_excel('golden_standard.xlsx')
print(f"Загружено {len(df)} размеченных отзывов")
print(f"Распределение классов:\n{df['is_fake'].value_counts()}")

# Берём только нужные колонки
df = df[['text', 'is_fake']].dropna()

# Разделяем на train/val/test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(),
    df['is_fake'].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df['is_fake'].tolist()
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print(f"\nTrain: {len(train_texts)}")
print(f"Validation: {len(val_texts)}")
print(f"Test: {len(test_texts)}")

MODEL_NAME = "cointegrated/rubert-tiny2"

print(f"\nЗагружаем модель {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "REAL", 1: "FAKE"},
    label2id={"REAL": 0, "FAKE": 1},
    ignore_mismatched_sizes=True
)

print(f"✅ Модель загружена. Параметров: {sum(p.numel() for p in model.parameters())}")

#Токенизация
class ReviewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewsDataset(train_texts, train_labels)
val_dataset = ReviewsDataset(val_texts, val_labels)
test_dataset = ReviewsDataset(test_texts, test_labels)

#Метрики
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir='./rubert_fake_detector',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Запуск обучения
print("\n" + "="*50)
print("НАЧАЛО ОБУЧЕНИЯ RuBERT")
print("="*50)

trainer.train()

# Оценка на тестовой выборке
print("\n" + "="*50)
print("ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*50)

test_results = trainer.evaluate(test_dataset)
print(f"\nРезультаты:")
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"F1-score: {test_results['eval_f1']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall: {test_results['eval_recall']:.4f}")

print("\n" + "="*50)
print("ПРОВЕРКА НА ПРИМЕРАХ")
print("="*50)

test_examples = [
    ("Очень крутое место! Всем рекомендую! Скидки каждый день!", "реклама"),
    ("Супер заведение, супер вкусно! Официант Андрей Андреевич - просто бомба! Были в пятницу вечером.", "реальный отзыв"),
    ("Нормальное кафе, можно поесть. Цены средние.", "реальный отзыв"),
]

model.eval()
for text, expected in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()

    print(f"\nТекст: {text[:80]}...")
    print(f"Предсказание: {'🔴 РЕКЛАМА' if pred == 1 else '🟢 РЕАЛЬНЫЙ'}")
    print(f"Вероятности: REAL={probs[0][0]:.2%}, FAKE={probs[0][1]:.2%}")
    print(f"Ожидание: {expected}")

model.save_pretrained('./fake_detector_model')
tokenizer.save_pretrained('./fake_detector_model')
print("\n✅ Модель сохранена в './fake_detector_model'")

Загружено 4004 размеченных отзывов
Распределение классов:
is_fake
1    3500
0     504
Name: count, dtype: int64

Train: 2802
Validation: 601
Test: 601

Загружаем модель cointegrated/rubert-tiny2...


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

✅ Модель загружена. Параметров: 29194394


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...


ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Результаты:
Accuracy: 0.9967
F1-score: 0.9981
Precision: 0.9962
Recall: 1.0000

ПРОВЕРКА НА ПРИМЕРАХ

Текст: Очень крутое место! Всем рекомендую! Скидки каждый день!...
Предсказание: 🔴 РЕКЛАМА
Вероятности: REAL=0.53%, FAKE=99.47%
Ожидание: реклама

Текст: Супер заведение, супер вкусно! Официант Андрей Андреевич - просто бомба! Были в ...
Предсказание: 🟢 РЕАЛЬНЫЙ
Вероятности: REAL=91.38%, FAKE=8.62%
Ожидание: реальный отзыв

Текст: Нормальное кафе, можно поесть. Цены средние....
Предсказание: 🟢 РЕАЛЬНЫЙ
Вероятности: REAL=98.61%, FAKE=1.39%
Ожидание: реальный отзыв


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Модель сохранена в './fake_detector_model'


In [ ]:

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Загружаем модель
print("Загружаем модель...")
tokenizer = AutoTokenizer.from_pretrained('./fake_detector_model')
model = AutoModelForSequenceClassification.from_pretrained('./fake_detector_model')
model.eval()

# Загружаем полный датасет
df_full = pd.read_csv('filtered_hotels_restaurants_cafes.csv')
print(f"Загружено {len(df_full)} отзывов")

if 'review' in df_full.columns and 'text' not in df_full.columns:
    df_full = df_full.rename(columns={'review': 'text'})

# Функция предсказания
def predict_fake(text):
    if pd.isna(text) or text == "":
        return 0, 0.0
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
        prob_fake = probs[0][1].item()
    return pred, prob_fake

# Применяем ко всем отзывам
print("\nПредсказываем для всех отзывов...")
tqdm.pandas()
df_full['is_fake'] = df_full['text'].progress_apply(lambda x: predict_fake(x)[0])
df_full['fake_probability'] = df_full['text'].progress_apply(lambda x: predict_fake(x)[1])

print(f"\nРезультаты:")
print(f"  - Реальных отзывов (0): {(df_full['is_fake'] == 0).sum()} ({(df_full['is_fake'] == 0).mean()*100:.1f}%)")
print(f"  - Рекламных отзывов (1): {(df_full['is_fake'] == 1).sum()} ({(df_full['is_fake'] == 1).mean()*100:.1f}%)")

# Сохраняем
df_full.to_csv('full_dataset_with_predictions.csv', index=False)
print("\n✅ Сохранено в 'full_dataset_with_predictions.csv'")

# Сохраняем только рекламные отзывы отдельно
df_ads = df_full[df_full['is_fake'] == 1]
df_ads.to_csv('detected_ads.csv', index=False)
print(f"✅ Сохранено {len(df_ads)} рекламных отзывов в 'detected_ads.csv'")

Загружаем модель...


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Загружено 136421 отзывов

Предсказываем для всех отзывов...


100%|██████████| 136421/136421 [30:27<00:00, 74.65it/s]



Результаты:
  - Реальных отзывов (0): 135708 (99.5%)
  - Рекламных отзывов (1): 713 (0.5%)

✅ Сохранено в 'full_dataset_with_predictions.csv'
✅ Сохранено 713 рекламных отзывов в 'detected_ads.csv'


In [ ]:
import pandas as pd

# Загружаем файл с предсказаниями
df = pd.read_csv('full_dataset_with_predictions.csv')

# Берём только фейковые (is_fake = 1)
fakes = df[df['is_fake'] == 1].copy()

# Сортируем по вероятности
fakes = fakes.sort_values('fake_probability', ascending=False)

print(f"Всего фейковых отзывов: {len(fakes)}")
print("\n" + "="*80)
print("30 ПРИМЕРОВ ФЕЙКОВЫХ (РЕКЛАМНЫХ) ОТЗЫВОВ:")
print("="*80 + "\n")

for i, (idx, row) in enumerate(fakes.head(30).iterrows(), 1):
    print(f"{i}. ВЕРОЯТНОСТЬ РЕКЛАМЫ: {row['fake_probability']:.2%}")
    print(f"   ТЕКСТ: {row['text']}")
    print("-"*80 + "\n")

Всего фейковых отзывов: 713

30 ПРИМЕРОВ ФЕЙКОВЫХ (РЕКЛАМНЫХ) ОТЗЫВОВ:

1. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.43%
   ТЕКСТ: Всё замечательно!!!! Рекомендую раз в недельку заходить! 

--------------------------------------------------------------------------------

2. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.42%
   ТЕКСТ: Фирменная пицца ETERNO - это фантастика! Спасибо! 

--------------------------------------------------------------------------------

3. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.42%
   ТЕКСТ: Супер!!!! Ела б каждый день 😁

--------------------------------------------------------------------------------

4. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.42%
   ТЕКСТ: Каждый год в феврале мы собираемся здесь! И всем советуем!!!

--------------------------------------------------------------------------------

5. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.42%
   ТЕКСТ: Чисто ! Аккуратно ! Все супер !!!

--------------------------------------------------------------------------------

6. ВЕРОЯТНОСТЬ РЕКЛАМЫ: 99.41%
   ТЕКСТ: Потрясающее место!! Приходили сюд